In [ ]:
import random
import json

from openai import OpenAI
from IPython.display import Markdown, display
from litellm import completion

openai = OpenAI()

participants = { "Alex": "gpt-4.1", "Blake":"claude-haiku-4-5", "Charlie": "gemini/gemini-2.5-flash-lite"}

def getSystemPrompt(speaker, listenerStr):
    system_prompt = f"""
    You are {speaker}, a chatbot who is very knowledgeable about Safaris in East Africa; you are brainstorming
    on where exactly you should go see them and at the same time have a wholesome cultural experience both
    out in the wild and in the city, in a clever and respectful way.
    You are in a conversation with {listenerStr}.
    """
    return system_prompt

def getUserPrompt(speaker, listenerStr):
    user_prompt = f"""
    You are {speaker}, in conversation with {listenerStr}.
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as {speaker}.
    Keep the responses short.
    """
    return user_prompt

chat = [{"speaker":"Alex", "content": "Hi there, we should go see the Big 5"}]
conversation = ''
print(chat[0]['speaker'])
speaker = chat[0]['speaker']
previous_speaker = chat[0]['speaker']
model = participants[speaker]

display(Markdown(f"### {speaker} (<small>*{model}*</small>) :\n{chat[0]['content']}\n"))

for i in range(5):
    pl = list(participants)
    listeners = [random.choice([p for p in pl if p != previous_speaker])] + [previous_speaker]
    listenerStr = " And ".join(listeners)

    speaker = list(set(participants) - set(listeners))[0]
    previous_speaker = speaker

    model = participants[speaker]
    system_prompt = getSystemPrompt(speaker, listenerStr)

    conversation = json.dumps(chat)
    user_prompt = getUserPrompt(speaker, listenerStr)

    messages = [{"role": "system", "content": system_prompt}]
    messages.append({"role": "user", "content": user_prompt})

    response = completion(model=model, messages=messages)
    content = response.choices[0].message.content.replace('"', '')
    chat.append({"speaker":speaker, "content": content})

    display(Markdown(f"### {speaker} (<small>*{model}*</small>) :\n{content}\n"))
